# 🚀 Huấn Luyện ConvMixer-512/12 (3.54M params) trên CIFAR-100 (Google Colab T4 GPU)

Notebook này dùng để tự kiểm chứng và huấn luyện **ConvMixer** (phiên bản nâng cấp 3.54M tham số) và **Res-ConvMixer** trên dataset **CIFAR-100**.

### 🎯 Thông số mô hình:
- **Kiến trúc**: ConvMixer-512/12 (12 tầng, 512 chiều đặc trưng)
- **Số tham số**: **3,542,628 (~3.54M params)**
- **Kích thước Patch**: $2 \times 2$ (256 tokens)
- **Bộ lọc không gian**: Depthwise Conv $7 \times 7$
- **Mục tiêu**: Bứt phá Val-Accuracy lên **~78% - 81%** trên CIFAR-100 (vượt xa mức 63% của ResMLP).

### Bước 1: Kiểm tra phần cứng GPU T4
Đảm bảo vào menu **Runtime** -> **Change runtime type** -> chọn **T4 GPU**.

In [ ]:
!nvidia-smi
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')

### Bước 2: Chuẩn bị Code dự án trên Colab
Kéo thả file `DebugTeam_Coding.zip` vào thanh thư mục bên trái của Colab rồi bấm chạy ô này:

In [ ]:
import os
# 1. Giải nén toàn bộ mã nguồn vào máy ảo Colab
!unzip -q -o /content/DebugTeam_Coding.zip -d /content/codebase
%cd /content/codebase

# 2. Cài đặt thư viện phụ thuộc
!pip install -q timm
!pip install -q -e .
print('✅ Đã giải nén và cài đặt xong thư viện!')

### Bước 3 (Tùy chọn - Chạy thử nghiệm nhanh trong 30 giây)
Chạy thử 2 epochs để tự kiểm chứng luồng dữ liệu, tính toán loss và accuracy trước khi chạy trọn vẹn:

In [ ]:
# Chạy thử nghiệm nhanh 2 epochs (~25 giây)
!python scripts/run_train.py --config configs/cifar100_conv_mixer.yaml --training.epochs 2

### Bước 4: Chạy Huấn Luyện Trọn Vẹn 100 Epochs (Mất ~20-25 phút)
Cấu hình chuẩn từ `configs/cifar100_conv_mixer.yaml`:
- `dim: 512, depth: 12, kernel_size: 7, patch_size: 2` (3.54M params)
- `batch_size: 128, optimizer: adamw, lr: 0.001, use_amp: true`

In [ ]:
# Huấn luyện ConvMixer-512/12 (3.54M tham số)
!python scripts/run_train.py --config configs/cifar100_conv_mixer.yaml

# (Tùy chọn) Nếu bạn muốn train bản kết hợp Res-ConvMixer (13.02M params):
# !python scripts/run_train.py --config configs/cifar100_res_conv_mixer.yaml

### Bước 5: Vẽ biểu đồ Loss & Accuracy
Hiển thị biểu đồ quá trình học tập sau 100 Epochs:

In [ ]:
import json, os, matplotlib.pyplot as plt
from pathlib import Path

metric_files = list(Path('runs').rglob('metrics.json'))
if metric_files:
    latest_metric = sorted(metric_files, key=os.path.getmtime)[-1]
    print(f'Đang đọc kết quả từ: {latest_metric}')
    with open(latest_metric) as f:
        history = json.load(f).get('history', [])
    
    epochs = [h['epoch'] for h in history]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(epochs, [h['train_loss'] for h in history], label='Train Loss', color='tab:blue')
    ax1.plot(epochs, [h['val_loss'] for h in history], label='Val Loss', color='tab:orange')
    ax1.set_title('CrossEntropy Loss')
    ax1.legend(); ax1.grid(True)
    
    accs = [h['val_top1'] for h in history]
    ax2.plot(epochs, accs, label='Val Top-1 Acc (%)', color='tab:green', linewidth=2)
    ax2.set_title(f'Validation Accuracy (Đỉnh: {max(accs):.2f}%)')
    ax2.legend(); ax2.grid(True)
    plt.tight_layout()
    plt.savefig('runs/convmixer_cifar100_plot.png', dpi=150)
    plt.show()
else:
    print('Chưa tìm thấy file metrics.json.')

### Bước 6: Tải về Checkpoint Mô Hình & Logs cho nhóm
Tải file zip chứa `best_model.pt`, `train.log`, `metrics.json` và `config.yaml` về máy Mac.

In [ ]:
# Nén toàn bộ thư mục runs chứa model weights và logs
!zip -r /content/results_convmixer_512.zip runs/

# Tải file zip về máy
from google.colab import files
files.download('/content/results_convmixer_512.zip')
print('✅ Đã đóng gói model, log và config để tải về!')